In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develop a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import numpy as np
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
from core.data import default_sessions

subj_id = "MR82"
sess_id = default_sessions["MR82"]

In [ ]:
"""--------------------------------------------"""
# add in model parameters to encoder, look at cvr2
# cvr2 with strategy model params
# reward prediction error (MF), q-learner, need to get q-value
"""---------------------------------------------"""
# build in interaction terms and see what pops out in the cvr2/dr2 plots
"""---------------------------------------------"""
# add movement over time
"""---------------------------------------------"""
# cvr2 across time
# --> when does encoding emerge across time?
"""---------------------------------------------"""
# aggregate across sessions
"""---------------------------------------------"""
# check for temporal autocorrelations by ensuring no encoding after shuffling trial data.
# incorporate as a sanity check to pass for all sessions

# > check the fits for different regularization constants
# define responsive
"""--------------------------------------------"""

In [ ]:
# TODO
# add movement svds to time resolved encoder
# reconstruct avg peth cond value with bweights as sanity check
# then time-resolved!

## init

In [ ]:
from sg.models import Encoder

encoder = Encoder(
    subj_id,
    sess_id,
    binwidth_ms=100,
    norm=True,
)
encoder.verify()

In [ ]:
n_svds = [2, 5, 10, 20, 50, 100, 200]

scores = np.zeros((len(n_svds) + 1, encoder.num_units))

scores[0] = encoder.scores["encoder"]

for i, n in enumerate(n_svds):
    encoder_svd = Encoder(
        subj_id,
        sess_id,
        norm=True,
        add_svd=True,
        num_svd=n,
        binwidth_ms=100,
    )
    encoder_svd.get_r2()

    scores[i + 1] = encoder_svd.scores["encoder"]

n_svds = np.concatenate(([0], n_svds))

In [ ]:
from scipy.stats import sem

m = scores.mean(axis=1)
s = sem(scores, axis=1)

fig, ax = plt.subplots(tight_layout=True)

ax.plot(n_svds, m)
ax.fill_between(n_svds, m - s, m + s, alpha=0.5)
ax.set_xticks(n_svds)
ax.set_xlabel("num svds")
ax.set_ylabel("average r2")

In [ ]:
from sg.models import Encoder, StrategyEncoder

encoder = Encoder(
    subj_id,
    sess_id,
    binwidth_ms=100,
    add_interaction=True,
    add_svd=True,
    num_svd=20,
    norm=True,
)
encoder.verify()

encoder_mb = StrategyEncoder(
    subj_id,
    sess_id,
    binwidth_ms=100,
    norm=True,
    add_interaction=True,
    add_svd=True,
    num_svd=20,
    strategy_params="mb",
    strategy_filter="mb",
)
encoder_mf = StrategyEncoder(
    subj_id,
    sess_id,
    binwidth_ms=100,
    norm=True,
    add_interaction=True,
    add_svd=True,
    num_svd=20,
    strategy_params="mf",
    strategy_filter="mf",
)

encoder_mb.verify()
encoder_mf.verify()

In [ ]:
from sg.models import ShuffledEncoder

se_mb = ShuffledEncoder(
    subj_id=subj_id,
    sess_id=sess_id,
    enc_class=StrategyEncoder,
    binwidth_ms=100,
    strategy_params="mb",
    balance_strategy=False,
    add_interaction=True,
    add_svd=True,
    num_svd=20,
    norm=True,
    strategy_filter="mb",
)
se_mf = ShuffledEncoder(
    subj_id=subj_id,
    sess_id=sess_id,
    enc_class=StrategyEncoder,
    binwidth_ms=100,
    strategy_params="mf",
    balance_strategy=False,
    add_interaction=True,
    add_svd=True,
    num_svd=20,
    norm=True,
    strategy_filter="mf",
)

se_mb.get_cvr2_all(n_iters=2)
se_mf.get_cvr2_all(n_iters=2)
se_mb.get_dr2_all(n_iters=2)
se_mf.get_dr2_all(n_iters=2)

In [ ]:
se_mb.cvr2_unit["response"].mean(axis=0)

In [ ]:
models_strat = {"mb": encoder_mb, "mf": encoder_mf}
tv_names = {strat: model.tv_names for (strat, model) in models_strat.items()}

In [ ]:
se_mb.regressors

In [ ]:
cvr2s = {
    strat: {
        reg: {
            regr: se.cvr2_unit[regr].mean(axis=0)[encoder.reg_idxs[reg]]
            for regr in se.regressors
        }
        for reg in encoder.regions
    }
    for strat, se in {"mb": se_mb, "mf": se_mf}.items()
}

dr2s = {
    strat: {
        reg: {
            regr: se.dr2_unit[regr].mean(axis=0)[encoder.reg_idxs[reg]]
            for regr in se.regressors
        }
        for reg in encoder.regions
    }
    for strat, se in {"mb": se_mb, "mf": se_mf}.items()
}

In [ ]:
from core.viz import plot_grouped_boxes
from utils.colors import colors_region

fig, axes = plt.subplots(
    ncols=2, nrows=2, figsize=(8, 7), sharey=True, tight_layout=True
)

for i, (strat, cvr2_strat) in enumerate(cvr2s.items()):
    for j, tv_type in enumerate(["base_regressors", "interaction"]):
        _, ax = plot_grouped_boxes(
            data={
                regr: {reg: cvr2_strat[reg][regr] for reg in encoder.regions}
                for regr in tv_names[strat][tv_type]
            },
            colors=colors_region,
            ylim=[-0.1, 0.8],
            ax=axes[i][j],
        )

        if i == 0:
            ax.set_title(tv_type)
        if j == 0:
            ax.set_ylabel(rf"cv $r^2$ distribution, {strat}")

fig.suptitle(subj_id)

### generate boxplots

In [ ]:
from sg.models import Encoder, StrategyEncoder

encoder = Encoder(
    subj_id,
    sess_id,
    binwidth_ms=100,
    norm=True,
)
encoder.verify()

encoder_mb = StrategyEncoder(
    subj_id,
    sess_id,
    binwidth_ms=100,
    strategy_params="mb",
    add_interaction=True,
    norm=True,
    strategy_filter="mb",
)
encoder_mf = StrategyEncoder(
    subj_id,
    sess_id,
    binwidth_ms=100,
    strategy_params="mf",
    add_interaction=True,
    norm=True,
    strategy_filter="mf",
)
encoder_mb.verify()
encoder_mf.verify()

In [ ]:
from core.data import subject_ids, session_ids
from sg.models import StrategyEncoder

weights = {strat: {reg: [] for reg in encoder.regions} for strat in ["mb", "mf"]}
for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
    e_mb = StrategyEncoder(
        subj_id,
        sess_id,
        binwidth_ms=100,
        strategy_params="mb",
        add_interaction=True,
        norm=True,
        strategy_filter="mb",
    )
    e_mf = StrategyEncoder(
        subj_id,
        sess_id,
        binwidth_ms=100,
        strategy_params="mf",
        add_interaction=True,
        norm=True,
        strategy_filter="mf",
    )

    try:
        e_mb.fit_encoder()
        e_mf.fit_encoder()
    except ValueError:
        continue

    assert np.all(e_mb.regions == e_mf.regions)
    for reg in e_mb.regions:
        weights["mb"][reg].append(e_mb.encoder_weights[e_mb.reg_idxs[reg]])
        weights["mf"][reg].append(e_mf.encoder_weights[e_mb.reg_idxs[reg]])

weights = {
    strat: {reg: np.concatenate(weights[strat][reg]) for reg in encoder.regions}
    for strat in ["mb", "mf"]
}

In [ ]:
models_strat = {"mb": encoder_mb, "mf": encoder_mf}
tv_idxs = {strat: model.tv_idxs for (strat, model) in models_strat.items()}
tv_names = {strat: model.tv_names for (strat, model) in models_strat.items()}

In [ ]:
from core.viz import plot_grouped_boxes
from utils.colors import colors_region
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR
# only plot task modulated

fig, axes = plt.subplots(
    ncols=3, nrows=2, figsize=(8, 7), sharey=True, tight_layout=True
)

for i, (strat, weights_strat) in enumerate(weights.items()):
    for j, tv_type in enumerate(["task_vars", "interaction", "strategy_params"]):
        _, ax = plot_grouped_boxes(
            data={
                regr: {
                    reg: weights_strat[reg][:, tv_idxs[strat][regr]]
                    for reg in encoder.regions
                }
                for regr in tv_names[strat][tv_type]
            },
            colors=colors_region,
            ylim=[-1, 1],
            ax=axes[i][j],
        )

        if i == 0:
            ax.set_title(tv_type)
        if j == 0:
            ax.set_ylabel(rf"$\beta$ distribution, {strat}")

fig.suptitle(subj_id)

save_fig(fig, FIGURES_DIR / "bweight" / "boxplot_comp", f"{subj_id}_bweight_comp.png")

In [ ]:
from core.data import get_model_params

model_params, _ = get_model_params(subj_id, sess_id)

In [ ]:
encoder_mb.trial_data

In [ ]:
plt.figure()
plt.plot(encoder_mb.trial_data["spe"])
plt.plot(encoder_mb.trial_data["bswitch_itrial"])
plt.xlabel("mf trials|")
plt.show()

In [ ]:
from core.viz import plot_scatter

plot_scatter(
    encoder_mf.encoder_weights[:, encoder_mf.tv_idxs["q_R"]],
    encoder_mf.encoder_weights[:, encoder_mf.tv_idxs["rpe"]],
    xlabel="q(L)",
    ylabel="q(R)",
)

## action-value 

In [ ]:
# add the following functions to data.py
# then add to load_sess
# then make the two strategy encoders and specify spe, b_L, b_R, rpe, q_L, q_R, etc.

## movement

In [ ]:
# more svds generally increases the alpha values
from sg.models import Encoder

encoder = Encoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
    ],
    binwidth_ms=100,
    add_interaction=True,
    norm=True,
)
encoder.verify()

In [ ]:
encoder.scores["encoder"].mean()

In [ ]:
from scipy.stats import sem

n_svds = np.array([0, 1, 2, 3, 4, 5, 10, 20, 50, 100, 200])
scores_mu = np.zeros(len(n_svds))
scores_sem = np.zeros(len(n_svds))

scores_mu[0] = encoder.scores["encoder"].mean()
scores_sem[0] = sem(encoder.scores["encoder"])


for i, n_svd in enumerate(n_svds):
    if n_svd == 0:
        continue

    e = Encoder(
        subj_id,
        sess_id,
        tv_keys=[
            "response",
            "rewarded",
        ],
        binwidth_ms=100,
        add_interaction=True,
        add_svd=True,
        num_svd=n_svd,
        norm=True,
    )
    e.get_r2()

    scores_mu[i] = e.scores["encoder"].mean()
    scores_sem[i] = sem(e.scores["encoder"])

In [ ]:
plt.figure(tight_layout=True)
plt.plot(n_svds, scores_mu)
plt.fill_between(n_svds, scores_mu - scores_sem, scores_mu + scores_sem, alpha=0.5)
plt.xlabel("no. svds")
plt.ylabel("average r2")
plt.show()

## akam 2021 fig 4

### nb

In [ ]:
(encoder.trial_data.strategy == 1).sample(frac=1).mean() == (
    encoder.trial_data.strategy == 1
).mean()

In [ ]:
reg = "DMS"
tt = "strategy"
control = True

if tt == "choice_reward":
    trial_type = {
        "lc": (encoder.trial_data.response == 1) & (encoder.trial_data.rewarded == 1),
        "rc": (encoder.trial_data.response == -1) & (encoder.trial_data.rewarded == 1),
        "li": (encoder.trial_data.response == 1) & (encoder.trial_data.rewarded == 0),
        "ri": (encoder.trial_data.response == -1) & (encoder.trial_data.rewarded == 0),
    }
elif tt == "strategy":
    if control:
        trial_type = {
            "mb": (encoder.trial_data.strategy == 1).sample(frac=1),
            "mf": (encoder.trial_data.strategy == -1).sample(frac=1),
        }
    else:
        trial_type = {
            "mb": (encoder.trial_data.strategy == 1),
            "mf": (encoder.trial_data.strategy == -1),
        }

psths_tt = {
    k: encoder.psths[reg][:, mask, :].mean(axis=1) for k, mask in trial_type.items()
}

In [ ]:
tt_mean = np.mean([psth for psth in psths_tt.values()], axis=0)

psths_tt = {k: psth - tt_mean for k, psth in psths_tt.items()}

In [ ]:
# [n_neurons, n_tt*n_tpoints]

psths_tt_mat = np.concatenate([psth.T for psth in psths_tt.values()])
print(psths_tt_mat.shape)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(psths_tt_mat)

plt.figure(tight_layout=True)
plt.plot(
    [np.sum(pca.explained_variance_ratio_[:i]) for i in range(psths_tt_mat.shape[0])]
)
plt.axhline(y=0.9, linestyle="--", color="#666666")
plt.xlabel("n. components")
plt.ylabel("explained variance ratio")
plt.show()

In [ ]:
psths_proj = pca.transform(psths_tt_mat)[:, :3]
psths_proj.shape

In [ ]:
n_tpoints = encoder.psths[reg].shape[-1]
psths_tt_proj = {
    k: psths_proj[i * n_tpoints : (i + 1) * n_tpoints, :]
    for i, k in enumerate(trial_type.keys())
}

In [ ]:
from core.viz import plot_trajectory
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors


def truncate_colormap(cmap, minval=0.35, maxval=1.0, n=256):
    if isinstance(cmap, str):
        cmap = plt.get_cmap(cmap)
    return mcolors.LinearSegmentedColormap.from_list(
        f"trunc({cmap.name},{minval:.2f},{maxval:.2f})",
        cmap(np.linspace(minval, maxval, n)),
    )


def plot_trajectories(
    psths, pc_a=0, pc_b=1, cmaps=None, cmap_min=0.35, ax=None, pad=0.05, **traj_kwargs
):
    if ax is None:
        _, ax = plt.subplots(figsize=(3, 3), tight_layout=True)

    all_xy = np.concatenate(list(psths.values()), axis=0)
    lim = np.abs(all_xy).max() * (1 + pad)

    default_cmaps = ["YlGn", "Blues", "OrRd", "RdPu"]
    if cmaps is None:
        cmaps = {k: default_cmaps[i] for i, k in enumerate(psths)}
    cmaps = {k: truncate_colormap(c, minval=cmap_min) for k, c in cmaps.items()}

    handles = []
    for k, psth in psths.items():
        plot_trajectory(
            psth[:, pc_a],
            psth[:, pc_b],
            xlabel=f"pc {pc_a + 1}",
            ylabel=f"pc {pc_b + 1}",
            cmap=cmaps[k],
            mn=-lim,
            mx=lim,
            ax=ax,
            **traj_kwargs,
        )
        handles.append(Line2D([0], [0], color=cmaps[k](0.6), lw=2, label=k))

    ax.legend(handles=handles)

    return ax


ax = plot_trajectories(psths_tt_proj, pc_a=0, pc_b=1)

### mb/mf

In [ ]:
from sklearn.decomposition import PCA


def get_psths_tt(encoder, tt="choice_reward", reg="DLS"):
    if tt == "choice_reward":
        trial_type = {
            "lc": (encoder.trial_data.response == 1)
            & (encoder.trial_data.rewarded == 1),
            "rc": (encoder.trial_data.response == -1)
            & (encoder.trial_data.rewarded == 1),
            "li": (encoder.trial_data.response == 1)
            & (encoder.trial_data.rewarded == 0),
            "ri": (encoder.trial_data.response == -1)
            & (encoder.trial_data.rewarded == 0),
        }
    elif tt == "strategy":
        trial_type = {
            "mb": (encoder.trial_data.strategy == 1),
            "mf": (encoder.trial_data.strategy == -1),
        }

    psths_tt = {
        k: encoder.psths[reg][:, mask, :].mean(axis=1) for k, mask in trial_type.items()
    }

    tt_mean = np.mean([psth for psth in psths_tt.values()], axis=0)
    psths_tt = {k: psth - tt_mean for k, psth in psths_tt.items()}
    psths_tt_mat = np.concatenate([psth.T for psth in psths_tt.values()])

    return psths_tt_mat


def akam_pca_fit(encoder, tt="strategy", reg="DLS"):
    psths_tt_mat = get_psths_tt(encoder, tt=tt, reg=reg)
    pca = PCA().fit(psths_tt_mat)

    return pca


def akam_pca_proj(encoder, pca, tt="strategy", reg="DLS"):
    psths_tt_mat = get_psths_tt(encoder, tt=tt, reg=reg)
    psths_proj = pca.transform(psths_tt_mat)[:, :5]

    n_tpoints = encoder.psths[reg].shape[-1]
    psths_tt_proj = {
        k: psths_proj[i * n_tpoints : (i + 1) * n_tpoints, :]
        for i, k in enumerate(trial_type.keys())
    }
    return psths_tt_proj


def akam_pca(encoder, reg="DLS"):
    pca = akam_pca_fit(encoder, reg=reg)
    psths_tt_proj = akam_pca_proj(encoder, pca=pca, reg=reg)
    return psths_tt_proj


reg = "DMS"
pca = akam_pca_fit(encoder, tt="strategy", reg=reg)
psths_tt_proj_mb = akam_pca_proj(encoder_mb, pca, tt="strategy", reg=reg)
psths_tt_proj_mf = akam_pca_proj(encoder_mf, pca, tt="strategy", reg=reg)

In [ ]:
def plot_trajectories_strategy(
    psths_mb,
    psths_mf,
    pc_mb=0,
    pc_mf=0,
    cmaps=None,
    cmap_min=0.35,
    ax=None,
    pad=0.05,
    **traj_kwargs,
):
    if ax is None:
        _, ax = plt.subplots(figsize=(3, 3), tight_layout=True)

    all_xy = np.concatenate(
        (
            np.concatenate(list(psths_mb.values()), axis=0),
            np.concatenate(list(psths_mf.values()), axis=0),
        )
    )
    lim = np.abs(all_xy).max() * (1 + pad)

    default_cmaps = ["YlGn", "Blues", "OrRd", "RdPu"]
    if cmaps is None:
        cmaps = {k: default_cmaps[i] for i, k in enumerate(psths_mb)}
    cmaps = {k: truncate_colormap(c, minval=cmap_min) for k, c in cmaps.items()}

    handles = []
    for k in psths_mb.keys():
        plot_trajectory(
            psths_mb[k][:, pc_mb],
            psths_mf[k][:, pc_mf],
            xlabel=f"pc {pc_mb + 1}, mb",
            ylabel=f"pc {pc_mf + 1}, mf",
            cmap=cmaps[k],
            mn=-lim,
            mx=lim,
            ax=ax,
            **traj_kwargs,
        )
        handles.append(Line2D([0], [0], color=cmaps[k](0.6), lw=2, label=k))

    ax.legend(handles=handles)

    return ax


pc = 0
plot_trajectories_strategy(psths_tt_proj_mb, psths_tt_proj_mf, pc_mb=pc, pc_mf=pc)

## psths

In [ ]:
from scipy.stats import zscore

reg = "DLS"


def plot_peths(reg="DLS", sort_by="peak_fr_time"):
    if sort_by == "peak_fr_time":
        psths_reg = encoder.psths[reg].mean(axis=1)
        psths_reg = zscore(psths_reg, axis=1)
        peak_fr_sort_idxs = np.argsort(
            [np.argsort(psths_reg[i])[-1] for i in range(len(psths_reg))]
        )

        plt.figure(figsize=(3, 4))
        plt.imshow(psths_reg[peak_fr_sort_idxs], cmap="magma")
        plt.xticks(
            np.arange(psths_reg.shape[1], step=10),
            np.round(encoder.tbin_edges[:-1:10], 2),
        )
        plt.colorbar(shrink=0.5)
        plt.title(f"{reg} PETHs (s.b. peak fr time)", fontsize=7)

        plt.show()


plot_peths(reg="DLS")
plot_peths(reg="DMS")

In [ ]:
from scipy.stats import pearsonr as r

bweight_diff = encoder_mb.encoder_weights - encoder_mf.encoder_weights
bweight_diff = bweight_diff[:, encoder.num_tents :]

diff_corr = np.array(
    [
        [
            r(bweight_diff[:, i], bweight_diff[:, j]).statistic
            for i in range(bweight_diff.shape[1])
        ]
        for j in range(bweight_diff.shape[1])
    ]
)
diff_corr_reg = {
    reg: np.array(
        [
            [
                r(
                    bweight_diff[encoder.reg_idxs[reg], i],
                    bweight_diff[encoder.reg_idxs[reg], j],
                ).statistic
                for i in range(bweight_diff.shape[1])
            ]
            for j in range(bweight_diff.shape[1])
        ]
    )
    for reg in encoder.regions
}

In [ ]:
def plot_corr(diff_corr, reg=None):
    if reg is not None:
        diff_corr = diff_corr[reg]
    fig, ax = plt.subplots(figsize=(5, 5))

    ax.imshow(diff_corr, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(diff_corr)), encoder.tv_keys, rotation=90)
    ax.set_yticks(range(len(diff_corr)), encoder.tv_keys)

    title = (
        rf"$r$ of bweight difference between strategies, {reg}"
        if reg is not None
        else r"$r$ of bweight difference between strategies"
    )
    fig.suptitle(title)


plot_corr(diff_corr)
plot_corr(diff_corr_reg, reg="DLS")
plot_corr(diff_corr_reg, reg="DMS")

In [ ]:
from core.data import get_strategy_filter_idxs


def build_encoder_strategy_ctrl(strategy="mb"):
    encoder = Encoder(
        subj_id,
        sess_id,
    )
    encoder.get_data()

    # get an even percentage of model-based and model-free
    idxs = get_strategy_filter_idxs(
        encoder.trial_data, balance_strategy=True, cond_balance=False
    )
    idxs_mb = np.random.choice(idxs["mb"], len(idxs["mb"]) // 2)
    idxs_mf = np.random.choice(idxs["mf"], len(idxs["mf"]) // 2)
    idxs = np.sort(np.concatenate((idxs_mb, idxs_mf)))

    encoder_strategy = StrategyEncoder(
        subj_id,
        sess_id,
        idxs=idxs,
        tv_keys=["response", "rewarded", "response_prev", "rewarded_prev"],
        strategy_filter=strategy,
    )
    encoder_strategy.fit_encoder()

    assert len(np.unique(encoder_strategy.trial_data["strategy"])) == 2
    assert np.isclose(
        (encoder_strategy.trial_data["strategy"] == 1).mean(), 0.5, atol=0.15
    ), (encoder_strategy.trial_data["strategy"] == 1).mean()

    return encoder_strategy


encoder_mb_ctrl = build_encoder_strategy_ctrl(strategy="mb")
encoder_mf_ctrl = build_encoder_strategy_ctrl(strategy="mf")

In [ ]:
bweight_diff_ctrl = encoder_mb_ctrl.encoder_weights - encoder_mf_ctrl.encoder_weights
bweight_diff_ctrl = bweight_diff_ctrl[:, encoder.num_tents :]

diff_corr_ctrl = np.array(
    [
        [
            r(bweight_diff_ctrl[:, i], bweight_diff_ctrl[:, j]).statistic
            for i in range(bweight_diff_ctrl.shape[1])
        ]
        for j in range(bweight_diff_ctrl.shape[1])
    ]
)
diff_corr_ctrl_reg = {
    reg: np.array(
        [
            [
                r(
                    bweight_diff_ctrl[encoder.reg_idxs[reg], i],
                    bweight_diff_ctrl[encoder.reg_idxs[reg], j],
                ).statistic
                for i in range(bweight_diff_ctrl.shape[1])
            ]
            for j in range(bweight_diff_ctrl.shape[1])
        ]
    )
    for reg in encoder.regions
}

In [ ]:
plot_corr(diff_corr_ctrl)
plot_corr(diff_corr_ctrl_reg, reg="DLS")
plot_corr(diff_corr_ctrl_reg, reg="DMS")

In [ ]:
diff_corr_emc = diff_corr - diff_corr_ctrl
diff_corr_emc_reg = {
    reg: diff_corr_reg[reg] - diff_corr_ctrl_reg[reg] for reg in encoder.regions
}

plot_corr(diff_corr_emc)
plot_corr(diff_corr_emc_reg, reg="DLS")
plot_corr(diff_corr_emc_reg, reg="DMS")

## weight correlation

In [ ]:
from core.data import tv_vals
from core.viz import plot_kdes

weight_diff = {}
for regr in encoder.tv_keys:
    if regr != "response_prev":
        regr_ = f"{regr}_{tv_vals[regr][0]}"
        weight_diff[regr_] = (
            encoder_mb.encoder_weights[:, encoder.dm_idxs[regr_]]
            - encoder_mf.encoder_weights[:, encoder.dm_idxs[regr_]]
        )

plot_kdes(weight_diff)

## pca on DMS/DLS robs

In [ ]:
import numpy as np

In [ ]:
from sklearn.decomposition import PCA
from utils.colors import colors_region

pca_dls = PCA().fit(encoder.robs[:, encoder.reg_idxs["DLS"]])
pca_dms = PCA().fit(encoder.robs[:, encoder.reg_idxs["DMS"]])
cum_var_dls = np.array([sum(pca_dls.explained_variance_ratio_[:n]) for n in range(100)])
cum_var_dms = np.array([sum(pca_dms.explained_variance_ratio_[:n]) for n in range(100)])

plt.figure(tight_layout=True)
plt.plot(
    cum_var_dls,
    color=colors_region["DLS"],
    label=f"DLS (n={np.where(cum_var_dls > 0.9)[0][0]})",
)
plt.plot(
    cum_var_dms,
    color=colors_region["DMS"],
    label=f"DMS (n={np.where(cum_var_dms > 0.9)[0][0]})",
)
plt.legend()
plt.axhline(y=0.9, color="#666666", linestyle="--")
plt.xlabel("n. components")
plt.ylabel("p(explained variance)")
plt.title("spike counts")
plt.show()

## pca based on weights

In [ ]:
from sklearn.decomposition import PCA

pca_dls = PCA().fit(encoder.encoder_weights[encoder.reg_idxs["DLS"]])
pca_dms = PCA().fit(encoder.encoder_weights[encoder.reg_idxs["DMS"]])

cum_var_dls = np.array(
    [
        sum(pca_dls.explained_variance_ratio_[:n])
        for n in range(encoder.num_tents + encoder.num_tv)
    ]
)
cum_var_dms = np.array(
    [
        sum(pca_dms.explained_variance_ratio_[:n])
        for n in range(encoder.num_tents + encoder.num_tv)
    ]
)

plt.figure(tight_layout=True)
plt.plot(
    cum_var_dls,
    color=colors_region["DLS"],
    label=f"DLS (n={np.where(cum_var_dls > 0.9)[0][0]})",
)
plt.plot(
    cum_var_dms,
    color=colors_region["DMS"],
    label=f"DMS (n={np.where(cum_var_dms > 0.9)[0][0]})",
)
plt.axhline(y=0.9, color="#666666", linestyle="--")
plt.xlabel("n. components")
plt.ylabel("p(explained variance)")
plt.title("encoding beta weights")
plt.legend()
plt.show()

In [ ]:
pca = PCA().fit(encoder.encoder_weights)
cum_var = np.array(
    [
        sum(pca.explained_variance_ratio_[:n])
        for n in range(encoder.num_tents + encoder.num_tv)
    ]
)

plt.figure(tight_layout=True)
plt.plot(cum_var)
plt.axhline(y=0.9, color="#666666", linestyle="--")
plt.xlabel("n. components")
plt.ylabel("p(explained variance)")
plt.show()

In [ ]:
n = np.where(cum_var >= 0.9)[0][0]
pca = PCA(n_components=n).fit(encoder.encoder_weights)
weights_lowd = pca.transform(encoder.encoder_weights)

In [ ]:
weights_lowd[:, :3]

In [ ]:
plt.figure()
plt.scatter(weights_lowd[:, 0], weights_lowd[:, 1], alpha=0.5, s=0.5)
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection="3d")

ax.scatter(xs=weights_lowd[:, 0], ys=weights_lowd[:, 1], zs=weights_lowd[:, 2], s=0.3)
plt.show()